# Clasificador de estilo en español

**Asignatura** · Análisis, Procesamiento y Aplicación de IA  
**Realizado por** · Joaquín Bermeo 
**Clases** · Formal e informal

El objetivo es evaluar si el ajuste fino de BETO aporta una mejora frente a TF-IDF con regresión logística al clasificar el registro de frases breves en español. Los modelos reciben solo la frase. Se comparan con F1-macro y accuracy sobre las mismas particiones.

Abra el notebook con el kernel de Python que tenga PyTorch instalado. En este proyecto puede seleccionar **Python 3.11 (kernel-laboratorios)** si tiene disponible el entorno venvDC. Ejecute las celdas en orden. La primera descarga de BETO requiere conexión. Una GPU acelera el ajuste. La celda de entrenamiento indica el dispositivo seleccionado y usa una carpeta temporal para sus checkpoints durante la sesión.

In [1]:
# Si faltan dependencias, descomente y ejecute la línea, vuelva a comentarla y reinicie el kernel.
# %pip install "transformers==5.22.0" "datasets>=4,<6" "accelerate>=1.1,<2" "scikit-learn>=1.4" "pandas>=2" "matplotlib>=3.7"
# Para PyTorch >=2.6 con CUDA, conserve su instalación compatible con la GPU.

In [2]:
from collections import Counter
from importlib.metadata import version
from time import perf_counter
import re
import sys
import unicodedata

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from sklearn.metrics import (
    ConfusionMatrixDisplay, accuracy_score, classification_report,
    confusion_matrix, f1_score,
)

SEMILLA = 42
CLASES = ["formal", "informal"]
IDS = list(range(len(CLASES)))
ID_A_ESTILO = dict(enumerate(CLASES))
ESTILO_A_ID = {estilo: i for i, estilo in ID_A_ESTILO.items()}
PARTICIONES = ["entrenamiento", "validación", "prueba"]

print("Python:", sys.version.split()[0])
print("Versiones:", {nombre: version(nombre) for nombre in (
    "torch", "transformers", "datasets", "accelerate", "scikit-learn"
)})

Python: 3.13.16
Versiones: {'torch': '2.14.1+cu130', 'transformers': '5.22.1', 'datasets': '4.8.5', 'accelerate': '1.15.0', 'scikit-learn': '1.6.1'}


## 1. Construcción del dataset

Se redactaron **800 frases sintéticas**, 400 por clase. Cada uno de los **400 pares** expresa una intención similar con dos registros. Hay 40 pares por tema y 10 temas. El tamaño aporta variedad de vocabulario, propósitos comunicativos y construcciones, y permite reservar 120 frases para cada conjunto de evaluación.

| Registro | Criterio de etiquetado |
|---|---|
| Formal | Redacción cuidada, propia de avisos, comunicaciones profesionales o exposiciones. Puede incluir tratamiento de usted, peticiones atenuadas, formulaciones impersonales o vocabulario preciso. |
| Informal | Redacción conversacional, propia de intercambios de confianza. Puede incluir tuteo, giros cotidianos, elipsis, énfasis o vocabulario coloquial. |

La cortesía, el tema y la puntuación pueden aparecer en ambas clases. La etiqueta se asigna según el registro predominante dentro de cada par. Hay **50 pares de contraste sutil** con observaciones sobre las marcas de registro. Sus etiquetas son decisiones de redacción para esta práctica. El corpus representa diez contextos escritos y no permite estimar el desempeño sobre todo el español espontáneo.

Cada línea del corpus incluye la versión formal, la informal y, cuando corresponde, una nota de contraste. Solo el texto y la etiqueta se utilizan en el ajuste.

In [3]:
# Una línea por par. La nota opcional se reserva para el análisis cualitativo.
CORPUS = {
    "Estudios": """
Se aceptarán las prácticas enviadas hasta las dieciocho horas.|Hasta las seis todavía puedes mandar la práctica.
¿Tendría disponibilidad para revisar el capítulo metodológico esta semana?|¿Tienes un rato esta semana para revisar la parte de metodología?
Los ejemplares prestados podrán devolverse en el buzón exterior.|Si vas a devolver los libros, déjalos en el buzón de afuera.
No es posible reconocer créditos sin el programa de la asignatura.|Sin el programa de la materia no te pueden contar esos créditos.
Le confirmo que la tutoría tendrá lugar en el aula del segundo piso.|Nos vemos para la tutoría en el salón de arriba, en el segundo piso.
¿Podría incorporar al acta los acuerdos de la última sesión?|¿Metes en el acta lo que acordamos la última vez?
El comité revisará la pertinencia de las referencias bibliográficas.|Los del comité van a ver si las fuentes sí vienen al caso.
Por favor, incluya su número de matrícula en la primera página.|Pon tu número de matrícula en la primera hoja, por favor.
Quienes opten por la modalidad virtual deberán activar el micrófono al intervenir.|Si entran en línea, prendan el micro cuando les toque hablar.
Se solicita indicar la fuente de las imágenes utilizadas.|Oigan, pongan de dónde sacaron las imágenes.
La credencial extraviada se encuentra en la oficina de coordinación.|Tu carnet perdido está en la oficina de coordinación.
Si requiere retroalimentación adicional, podemos programar una segunda revisión.|Si quieres más comentarios, nos juntamos otra vez y lo vemos.
Les pido que dejen libre el pasillo al finalizar la sesión.|Chicos, cuando acaben no dejen cosas tiradas en el pasillo.
El promedio del módulo se calculará sin considerar la actividad diagnóstica.|La actividad de diagnóstico no te cuenta para el promedio del módulo.|Ambas conservan «promedio» y «módulo». La forma «se calculará sin considerar» es impersonal y formal. «No te cuenta» se dirige a quien lee.
Les remito la versión corregida de la rúbrica para su consulta.|Les paso la rúbrica arreglada para que la tengan a mano.
¿Sería tan amable de reservar un proyector para la exposición del jueves?|¿Me guardas un proyector para la expo del jueves?
Solicito que se rectifique la fecha indicada en mi historial académico.|Quiero que me arreglen la fecha que me sale en el historial de notas.
La consulta de calificaciones se habilitará al cierre del semestre.|Apenas acabe el semestre ya vas a poder ver tus notas.
Agradezco que me haya señalado el error en la cita.|Gracias por pillarte el error de la cita y avisarme.
La sala de lectura permanecerá abierta durante el receso.|En el descanso igual puedes quedarte leyendo en la sala.
Conviene conservar la escala original al reproducir el gráfico.|Mejor deja la escala como estaba cuando copies el gráfico.
La entrega grupal requiere una sola copia por equipo.|Con que uno del grupo suba el trabajo ya está.
Le agradeceré que identifique al autor de cada intervención.|¿Pones quién dijo cada cosa? Gracias.
El taller de redacción se trasladó al bloque contiguo.|El taller de escritura es ahora en el bloque de al lado.
El ejercicio inicial servirá para identificar los conocimientos previos.|Lo primero es un ejercicio para ver qué tanto ya saben.
El acceso al repositorio se habilitará una vez verificado el correo institucional.|Cuando te comprueben el correo de la U, ya puedes entrar al repositorio.
La lista definitiva incluirá a quienes hayan confirmado su asistencia.|En la lista final solo van los que dijeron que sí van.
Le presento las disculpas correspondientes por la omisión de su nombre.|Perdón, se me pasó poner tu nombre.
Podemos analizar la propuesta en la próxima reunión de coordinación.|Eso lo podemos ver juntos en la próxima reunión.
El apartado de resultados requiere una descripción de las variables incluidas.|En resultados falta decir qué variables metiste.
La exposición se limitará a cinco minutos por participante.|Cada uno tiene cinco minutos para hablar, pilas con el tiempo.
¿Podría reservar el espacio hasta que concluya la defensa?|¿Nos dejas usar el salón hasta que acabe la defensa?
Las consultas sobre el temario se recibirán por el foro del curso.|Si algo del temario no te queda claro, escríbelo en el foro.
El formato de inscripción estará disponible en la página del departamento.|En la página del departamento está la ficha para que te apuntes.
La revisión quedó fijada para el miércoles a mediodía.|El miércoles al mediodía vemos qué tal quedó.|La fecha se conserva. La expresión institucional «quedó fijada» contrasta con «vemos qué tal quedó», propia de un intercambio entre compañeros.
La pregunta fue reformulada para precisar su alcance.|Le cambiamos un poco la pregunta para que se entienda hasta dónde llega.
Corresponde presentar una justificación por cada ausencia registrada.|Por cada vez que faltaste te toca llevar un justificativo.
Si le resulta conveniente, podemos conversar al término de la clase.|Si te queda bien, hablamos saliendo de clase.
La fecha de publicación sigue pendiente de confirmación.|Todavía no sabemos cuándo lo suben.
Quedamos a su disposición para resolver cualquier consulta sobre el módulo.|Si algo del módulo te deja con dudas, nos dices y lo vemos.
""",
    "Trabajo": """
¿Podría actualizar el inventario antes de la visita del proveedor?|¿Me dejas al día el inventario antes de que venga el proveedor?
El equipo técnico confirmó que la interrupción se originó en el servidor central.|Los de sistemas dijeron que todo se cayó por el servidor principal.
Remito la cotización con el desglose de los costos de instalación.|Te paso la cotización con lo que cuesta cada parte del montaje.
La inducción de personal tendrá lugar el martes en la sede norte.|El martes les explican el trabajo a los nuevos en la oficina del norte.
Agradecemos que remita las observaciones por este mismo medio.|Mándanos lo que veas raro por aquí mismo, gracias.
El cliente solicitó que la entrega se realizara a primera hora.|El cliente pidió que se lo llevemos bien tempranito.
La orden de compra se emitirá una vez recibido el aval de gerencia.|Apenas gerencia dé el visto bueno, sacamos la orden de compra.
Le agradecería precisar cuáles son los cambios prioritarios.|¿Me dices qué cambios van primero?
Es necesario registrar el incidente antes de cerrar el turno.|Anota lo que pasó antes de que te vayas del turno.
La sucursal dejó de recibir despachos los sábados.|A esa sucursal ya no le mandan nada los sábados.
El plazo interno se prorrogó hasta el cierre del mes.|Nos dieron chance de terminar hasta fin de mes.
El reemplazo asumirá las funciones a partir del próximo lunes.|El lunes que viene ya entra la persona que te va a cubrir.
Le envío el contrato para la revisión de las cláusulas de renovación.|Te mando el contrato para que mires cómo quedó lo de renovarlo.
¿Sería posible verificar el monto antes de autorizar el depósito?|¿Puedes chequear cuánto es antes de dar luz verde al depósito?
El ingreso a la planta requiere calzado de seguridad.|Para entrar a la planta tienes que ir con botas de seguridad.
Se adjunta el balance correspondiente al tercer trimestre.|Aquí va el balance de esos tres meses.
La llamada con el distribuidor quedó pendiente por un problema de conexión.|Se fue la conexión y nos faltó hablar con el distribuidor.
Las pruebas del sistema se completaron dentro del plazo previsto.|Sí alcanzamos a hacer las pruebas del sistema a tiempo.|El contenido técnico permanece. La forma impersonal «se completaron» y «plazo previsto» contrastan con «sí alcanzamos» y «a tiempo».
El proveedor aún no ha emitido el comprobante solicitado.|El proveedor todavía no nos manda el comprobante que le pedimos.
¿Podría precisar desde qué fecha se aplica la nueva tarifa?|¿Desde cuándo nos van a cobrar esa tarifa nueva?
Solicito autorización para intercambiar el turno del viernes.|¿Me dan permiso de cambiar el turno del viernes con alguien?
La reunión de seguimiento quedó fijada para las diez.|Nos vemos a las diez para ver cómo va todo.
El informe se incorporó al expediente del proyecto.|Ya dejamos el informe entre los papeles del proyecto.
Puede retirar su credencial en el área de recepción.|Pasa por recepción a recoger tu credencial.
Si el ajuste resulta viable, se aplicará en la próxima actualización.|Si da para hacer ese cambio, lo metemos en la próxima versión.
A partir de hoy, la solicitud de insumos se hará a través del formulario interno.|Desde hoy pidan lo que les haga falta con el formulario de la empresa.
El área contable requiere respaldo para el gasto de combustible.|Los de contabilidad necesitan el recibo de la gasolina.
La impresora permanecerá fuera de servicio mientras se sustituye el cabezal.|No usen la impresora todavía, que le están cambiando el cabezal.
La jefatura ha recibido su observación sobre el cronograma.|La jefa ya vio lo que dijiste del cronograma.
Me gustaría consultar su disponibilidad para una breve entrevista.|¿Tienes un ratito para que te haga unas preguntas?
La autorización anterior conserva su vigencia hasta mañana.|El permiso que ya tenías te sirve hasta mañana.
El documento debe quedar listo para la revisión del directorio.|Hay que dejar esto listo para que lo vea el directorio.
La sala grande quedó reservada para la capacitación de la tarde.|En la tarde usamos la sala grande para la capacitación, ya está apartada.
Los materiales restantes deberán almacenarse en el estante inferior.|Lo que sobre déjenlo en la repisa de abajo.
La sesión de trabajo se iniciará una vez que se incorporen todos los participantes.|Arrancamos cuando ya estén todos conectados.
¿Podría notificar al personal que la prueba de alarmas se adelantó?|Avísales que van a probar las alarmas más temprano, ¿sí?
La revisión del manual se encuentra en su etapa final.|Ya casi terminamos de revisar el manual.
Se habilitó una segunda fecha para las entrevistas de selección.|Abrieron otro día para hacer las entrevistas.
El viernes se comunicará la decisión sobre la renovación.|El viernes nos dicen si al final renuevan o no.|Ambas empiezan con el mismo día y anuncian el resultado. La primera nominaliza la decisión y la segunda la expresa como una pregunta indirecta en presente.
Puedo atenderle el jueves si desea revisar el presupuesto conmigo.|Si quieres que veamos juntos el presupuesto, el jueves puedo.
""",
    "Compras y servicios": """
El importe se acreditará en su cuenta dentro de cinco días hábiles.|En unos cinco días laborables ya te cae la plata en la cuenta.
Lamentamos que el artículo no corresponda a lo indicado en la compra.|Qué pena que te haya llegado algo distinto de lo que compraste.
Para gestionar la garantía, adjunte una fotografía de la etiqueta.|Para ver lo de la garantía, mándanos una foto de la etiqueta.
La reparación del cierre tiene un costo adicional.|Por arreglarte el cierre te cobran un poquito más.
La solicitud de cancelación debe enviarse antes del siguiente cobro.|Pide la baja antes de que te vuelvan a cobrar.
El comprobante incluye el descuento aplicado en caja.|En el recibo puedes ver cuánto te rebajaron en caja.
¿Podría indicarme si el paquete admite cambios de destino?|¿Sabes si todavía puedo cambiar a dónde mandan el paquete?
Su reserva podrá mantenerse hasta quince minutos después de la hora acordada.|Te guardamos la mesa hasta un cuarto de hora después de la hora que dijiste.
En caso de no recibir el código, solicite un nuevo envío.|Si el código no te llega, dale otra vez a enviar.
Hemos registrado la dirección actualizada en su ficha.|Listo, ya te cambiamos la dirección en la ficha.
El local dispone de un acceso lateral para el retiro de pedidos.|Por el costado del local puedes recoger lo que pediste.
Le solicito una copia legible del recibo de la transferencia.|Mándame una foto del recibo de la transferencia donde se lea bien, porfa.
La inscripción al servicio es gratuita durante el primer mes.|El primer mes te puedes apuntar sin pagar nada.
El número de rastreo se genera al despachar el envío.|Apenas salga el paquete, te dan el número para seguirlo.
Solicito que el reembolso se realice en el medio de pago original.|Que me devuelvan lo que pagué a la misma tarjeta, por favor.
Podemos ofrecerle un producto equivalente por el mismo importe.|Te podemos dar otro parecido por lo mismo que pagaste.
La tarifa publicada incluye el servicio de instalación a domicilio.|Con ese precio ya te lo van a instalar en casa.
¿Tendría la amabilidad de confirmar la talla antes del envío?|Antes de mandarlo, ¿me confirmas bien qué talla quieres?
La entrega quedó reprogramada para el siguiente día laborable.|Te cambiaron la entrega al siguiente día de trabajo.
Requerimos su autorización para modificar el titular del servicio.|Necesitamos que nos digas que sí para poner el servicio a nombre de otra persona.
El saldo disponible figura al pie de la factura.|Lo que te queda de saldo sale abajo en la factura.
La entrega a domicilio está disponible únicamente dentro del perímetro urbano.|Solo te lo llevan a la casa si estás dentro de la ciudad.
Recibimos su solicitud de cambio y se encuentra en revisión.|Ya nos llegó tu pedido de cambio y lo estamos viendo.
Queda pendiente la validación de los datos de facturación.|Falta comprobar que los datos de la factura estén bien puestos.
El personal de recepción le indicará dónde retirar las llaves.|En recepción te dicen de una dónde recoger las llaves.
Le confirmaré el número de mesa en cuanto disponga de esa información.|Apenas sepa qué mesa es, te aviso.
El sistema aplicó el cargo dos veces por la misma operación.|Me cobraron dos veces por lo mismo.|Las dos describen un cobro duplicado. La primera identifica al sistema con «aplicó el cargo». La segunda adopta la perspectiva del cliente con «me cobraron».
¿Aceptaría recibir el pedido en dos entregas diferentes?|¿Te sirve si te lo mandamos en dos partes?
La devolución se autoriza si conserva el embalaje original.|Si aún tienes la caja en la que vino, te dejan devolverlo.
La fecha de retiro consta en el resguardo de la reparación.|En el papelito de la reparación dice cuándo pasar a recogerlo.
Se aplicará un ajuste en la próxima factura por los días sin servicio.|Te descuentan en la siguiente factura los días que te quedaste sin servicio.
Necesitamos que especifique qué repuesto desea adquirir.|Dinos qué pieza quieres comprar, que hay varias.
La orden permanece disponible para retiro durante toda la semana.|Puedes ir por tu pedido cualquier día de esta semana.
Me comuniqué con la tienda para consultar por el cambio.|Llamé a la tienda para ver qué onda con el cambio.
El artículo se encuentra reservado a nombre de la persona que hizo el pedido.|Lo tienen apartado a nombre de quien lo pidió.|Comparten «a nombre de». La versión breve usa un pronombre con referente implícito y «apartado». La formal identifica el artículo explícitamente.
Si la pieza no está disponible, le ofreceremos una alternativa de características similares.|Si no queda esa pieza, buscamos una parecida para ti.
Agradezco la atención brindada durante la gestión de mi reclamo.|Gracias por ayudarme con todo el lío del reclamo.
¿Dispone de una factura electrónica para esta transacción?|¿Tienes la factura en digital de esta compra?
Le confirmo que la oficina atiende con cita previa.|Te aviso que para ir a esa oficina hay que sacar cita primero.
La consulta por el estado de su pedido puede realizarse en línea.|Puedes mirar por internet dónde va tu pedido.
""",
    "Hogar": """
Le agradecería que mantuviera la ventana cerrada durante la noche.|¿Dejas cerrada la ventana en la noche, porfa?
Propongo distribuir entre todos las tareas pendientes de la casa.|Repartámonos lo que falta hacer en la casa.
La llave de repuesto se encuentra en el cajón de la entrada.|Dejé la otra llave en el cajón junto a la puerta.
Le agradeceré que retire la ropa del tendedero si empieza a llover.|Si se viene la lluvia, entra la ropa, ¿sí?
El portero recogerá la correspondencia durante mi ausencia.|Mientras no esté, el conserje me guarda las cartas.
Le aviso que el técnico revisará el calefón al mediodía.|Oye, al mediodía viene el señor a ver qué le pasa al calefón.
Sugiero conservar la sopa refrigerada hasta el momento de servirla.|Mete la sopa en la refri y sácala cuando toque comer.
¿Podría indicarme cuándo desocupará la lavadora?|¿A qué hora queda libre la lavadora?
Lamento que los trabajos de pintura hayan producido tanto ruido.|Perdón por la bulla que hicimos pintando.
Agradezco que se haya ocupado de regar las plantas.|Gracias por acordarte de echarles agua a las plantas.
Se recomienda dejar libre el espacio junto al medidor.|No pongas nada al lado del medidor, mejor.
El contrato de arriendo vence a finales de noviembre.|Se nos acaba el arriendo a fines de noviembre.|Ambas expresan el mismo vencimiento. «Se nos acaba» involucra a quien habla y suena conversacional. La otra frase identifica el contrato y usa «vence».
La humedad del dormitorio requiere una inspección adicional.|Toca volver a ver de dónde sale la humedad del cuarto.
Si está de acuerdo, puedo encargarme de recibir los muebles.|Si te parece, yo me quedo a recibir los muebles.
Por seguridad, asegure el pestillo antes de retirarse.|Cuando te vayas, ponle el seguro a la puerta.
El horno debe precalentarse antes de introducir la bandeja.|Prende el horno un rato antes de meter la lata.
Solicito su autorización para instalar un estante en el corredor.|¿Me dejas poner una repisa en el pasillo?
Esta noche se esperan temperaturas inferiores a las de ayer.|Hoy va a hacer más frío en la noche que ayer.|Las dos son descriptivas. La primera adopta la voz impersonal de un pronóstico. La segunda usa «va a hacer más frío», habitual en conversación.
El detergente se agotó después del último lavado.|Con la última lavada ya no quedó jabón.
Le confirmo que mañana recibiremos la visita del administrador.|Te cuento que mañana se pasa el administrador.
Si necesita un espacio para estacionar, puede usar el mío.|Si no tienes dónde dejar el carro, usa mi puesto.
La cuota del edificio incluye la limpieza de las zonas comunes.|En lo que pagamos del edificio ya va la limpieza de los espacios de todos.
El recipiente debe permanecer cerrado para evitar derrames.|Ciérralo bien, que si no se riega todo.
¿Le sería posible recoger al perro antes del anochecer?|¿Puedes ir por el perro antes de que oscurezca?
Le pido que evite dejar residuos junto a la escalera.|Porfa, no dejes la basura al lado de las gradas.
El piso no estará seco hasta dentro de media hora.|Dale media horita antes de pisar, que todavía está mojado.
Podemos revisar el gasto de electricidad cuando llegue la factura.|Cuando venga la planilla vemos cuánto gastamos en luz.
El teléfono de contacto está anotado junto al intercomunicador.|Anoté el número al lado del citófono, por si lo necesitas.
La compra de víveres quedó prevista para el sábado por la mañana.|El sábado temprano vamos a hacer las compras.
Por favor, compruebe que la hornilla quedó apagada.|Mira si quedó apagada la hornilla, por favor.
Necesito confirmar cuántas personas asistirán a la cena.|¿Al final cuántos vienen a comer en la noche?
La cuna se colocará en la habitación contigua.|Vamos a poner la cuna en el cuarto de al lado.
Se guardaron los alimentos sobrantes para el día siguiente.|Lo que sobró de comer lo guardé para mañana.
Le devolveré el taladro tan pronto termine la instalación.|Apenas acabe de poner eso, te devuelvo el taladro.
La entrega de gas se realizará en la entrada principal.|Lo del gas lo dejan en la puerta de adelante.
El perro permanece en el patio mientras se ventila la sala.|Dejé al perro afuera mientras se airea la sala.
Le agradecería informar a los demás que mañana habrá visita.|¿Les cuentas a los demás que mañana viene gente?
Conviene sustituir la bombilla antes de que deje de funcionar por completo.|Mejor cambia ese foco antes de que se dañe del todo.
La caja de herramientas está disponible en el cuarto de almacenamiento.|Las herramientas están en la bodeguita si te hacen falta.
Le agradezco que haya mantenido despejada la entrada.|Gracias por dejar libre la entrada, me hiciste un favor.
""",
    "Movilidad": """
El próximo autobús saldrá del andén siete.|El bus que sigue sale del andén siete, pilas.
¿Podría indicarme cuánto tiempo tarda el trayecto hasta la terminal?|¿Sabes cuánto se demora uno en llegar a la terminal?
El equipaje de mano debe colocarse debajo del asiento delantero.|Pon tu mochila debajo del asiento de enfrente.
La vía alterna permanece habilitada para vehículos livianos.|Si vas en carro, todavía puedes pasar por el otro camino.
Recomiendo adquirir los pasajes con un día de antelación.|Mejor compra los pasajes desde el día anterior.
Le confirmo que el conductor ya se dirige a la dirección indicada.|El chofer ya va para donde le dijiste.
Por razones de mantenimiento, el ascensor de la estación está fuera de servicio.|Están arreglando el ascensor de la estación, toca ir por las gradas.
El valor del boleto no incluye el traslado desde el hotel.|Con el pasaje no te pagan el viaje desde el hotel.
¿Podría cambiarme a un asiento junto a la ventana?|¿Me cambias a un puesto al lado de la ventana?
El acceso a la ciclovía se encuentra al final de la calle.|Sigue hasta donde acaba la calle y ahí entras a la ciclovía.
La plataforma de embarque se informará poco antes de la salida.|Casi al salir nos dicen por dónde hay que subir.
El carril derecho quedará reservado para el transporte público.|Por el carril de la derecha van a pasar solo los buses.
El recorrido se modificó por el cierre de un puente.|Cerraron un puente y por eso toca ir por otro lado.
Agradezco que me haya avisado del desvío con anticipación.|Gracias por avisarme del desvío a tiempo, así no me perdí.
Le sugiero esperar bajo el techo hasta que llegue el vehículo.|Mejor espéralo ahí donde hay techo hasta que llegue.
El vuelo llegará con una demora estimada de cuarenta minutos.|El avión va a llegar unos cuarenta minutos tarde.|La estimación es idéntica. «Demora estimada» y el futuro simple concentran el contraste frente a «va a llegar» y «unos cuarenta minutos tarde».
Los ciclistas deberán descender para cruzar el paso peatonal.|Si vas en bici, bájate para cruzar por el paso.
La tarifa nocturna se aplica después de las veintidós horas.|Después de las diez de la noche te cobran la tarifa nocturna.
La reserva permite transportar una bicicleta sin costo adicional.|Con esa reserva llevas la bici gratis.
Si lo prefiere, puedo dejarle en la entrada del parque.|Si quieres, te dejo justo en la entrada del parque.
El conductor confirmó la disponibilidad de espacio para la silla plegable.|El chofer dijo que sí cabe la silla plegable.
La conexión con el tren regional requiere un cambio de andén.|Para coger el tren regional toca cambiar de andén.
Por favor, valide la tarjeta antes de ingresar al vagón.|Pasa la tarjeta por la máquina antes de subirte, porfa.
Para salir del estacionamiento, conserve el boleto hasta llegar a la barrera.|No botes el ticket todavía, que te sirve para salir del parqueadero.
El puente permanecerá cerrado hasta que se retire el vehículo averiado.|No abren el puente hasta que saquen el carro que se quedó botado.
Me comunicaré con usted al llegar al punto de encuentro.|Te escribo cuando esté donde quedamos.
Es posible continuar a pie por la ruta señalizada.|Si sigues los letreros, puedes ir a pie por ahí.
La agencia reprogramó la excursión por falta de cupos.|No había puestos, así que cambiaron de día el paseo.
La salida prevista para el domingo requiere confirmación previa.|Hay que decir antes si vas al paseo del domingo.
El cruce permanecerá habilitado mientras duren las obras.|Aunque estén arreglando, igual se puede cruzar por ahí.
La bicicleta alquilada se devuelve en cualquiera de las estaciones.|Deja la bici que alquilaste en la estación que te quede mejor.
Se ruega esperar el descenso de los pasajeros antes de abordar.|Deja que se bajen primero y de ahí subes.
El servicio adicional solo opera los días feriados.|Ese otro bus solo pasa en feriados.
No fue posible registrar el equipaje dentro del horario establecido.|Llegamos tarde para dejar las maletas en el mostrador.
La unidad se detendrá únicamente en las paradas autorizadas.|El bus para solo donde hay parada, no en cualquier esquina.
La aplicación actualizará el punto de recogida automáticamente.|La app solita te cambia dónde te van a recoger.
El mapa indica una distancia aproximada de dos kilómetros.|Según el mapa quedan como dos kilómetros.|Ambas citan la misma fuente y distancia. El valor aproximado se expresa con «distancia aproximada» en una y con el adverbio conversacional «como» en la otra.
Por favor, informe al conductor si precisa ayuda con el equipaje.|Si necesitas una mano con las maletas, dile al chofer.
Se habilitó una zona de espera para quienes viajen con mascotas.|Hay un sitio aparte para esperar si vas con tu mascota.
Quisiera confirmar si la parada próxima al hospital sigue operativa.|¿La parada que queda por el hospital todavía funciona?
""",
    "Cultura": """
La función dará comienzo cuando se apaguen las luces de la sala.|Apenas apaguen las luces ya empieza la función.
Le agradecería que confirmara cuántos instrumentos traerá el grupo.|¿Me confirmas cuántos instrumentos van a traer?
El catálogo incluye una reseña de cada obra expuesta.|En el catálogo te cuentan un poquito de cada obra.
La grabación del espectáculo requiere el permiso de la organización.|Antes de grabar el show, pídele permiso a los que organizan.
La primera edición del libro se agotó en pocas semanas.|Los primeros ejemplares volaron en unas semanas.
Podemos ofrecerle una ubicación más próxima al escenario.|Te podemos dar un puesto más cerquita del escenario.
Se admitirá el ingreso del público durante el intermedio.|Si llegas tarde, aprovecha el descanso para entrar.
La presentación contará con la participación de la autora.|La autora también va a estar en la presentación.|El par mantiene la autora y la presentación. «Contará con la participación» tiene tono de programa oficial y «también va a estar» suena a comentario cotidiano.
Agradezco la invitación a colaborar con la selección de fotografías.|Gracias por invitarme a escoger las fotos con ustedes.
La exposición itinerante visitará tres localidades durante el verano.|La expo va a pasar por tres lugares este verano.
Se recomienda leer el prólogo para conocer el contexto de la obra.|Léete el prólogo, ahí entiendes de dónde salió el libro.
Le ruego abstenerse de utilizar el flash dentro de la galería.|Si vas a sacar fotos en la galería, sin flash, ¿sí?
¿Podría reservar dos localidades para la función vespertina?|¿Me guardas dos puestos para la función de la tarde?
La pieza final se interpretará con un acompañamiento de cuerdas.|La última canción va con cuerdas de fondo.
El público podrá conversar con el elenco al terminar la función.|Cuando se acabe la obra puedes quedarte a hablar con los actores.
El préstamo de partituras se registra en la recepción del teatro.|Si te llevas una partitura prestada, anótalo en recepción.
Puedo remitirle el programa si facilita una dirección de correo.|Pásame tu correo y te mando el programa.
La proyección se realizará en versión original con subtítulos.|La peli va en el idioma original y con subtítulos.
El taller de cerámica dispone de materiales para todos los inscritos.|Si te apuntas a cerámica, ahí ya te dan los materiales.
Conviene revisar la duración del documental antes de programar la proyección.|Mira cuánto dura el documental antes de ponerle hora.
Su asiento está ubicado en el extremo izquierdo de la tercera fila.|Tu puesto queda al final del lado izquierdo de la tercera fila.|Se conservan fila y lado. El posesivo «su» y «está ubicado» contrastan con el tuteo de «tu» y el verbo «queda».
La restauración de la escultura concluyó antes de la inauguración.|Terminaron de arreglar la escultura justo antes de inaugurar.
Las inscripciones para el coro se reciben durante el mes de abril.|En abril te puedes anotar para cantar en el coro.
¿Sería posible incluir el nombre del fotógrafo junto a la imagen?|¿Puedes poner quién tomó la foto al ladito?
El montaje incorporó escenas que no figuraban en el guion inicial.|En la versión final metieron escenas que antes no estaban en el guion.
Si desea participar en la lectura, comuníquese con la persona encargada.|Si te animas a leer, habla con quien está organizando.
El museo ofrece visitas guiadas para grupos de hasta doce personas.|En el museo te hacen el recorrido si son doce o menos.
La biblioteca exhibirá los manuscritos durante dos semanas.|En la biblioteca van a mostrar esos escritos por dos semanas.
La crítica destacó el uso de la iluminación en el segundo acto.|Lo que más gustó de la segunda parte fue cómo usaron las luces.
Le confirmo que las pruebas de sonido finalizaron esta mañana.|Ya acabaron de probar el sonido hoy en la mañana, te aviso.
La publicación digital conserva el orden de la edición impresa.|La versión que lees en pantalla va en el mismo orden que la de papel.
Solicito acceso al archivo audiovisual para preparar la muestra.|¿Me dejan entrar al archivo de videos para armar la muestra?
La sección infantil se ubicará cerca de la entrada del recinto.|Lo de los niños va a estar cerquita de la entrada.
La orquesta interpretará la obra en su tonalidad original.|Van a tocar la pieza en el tono en el que salió.
Lamentamos que se haya reducido el número de localidades disponibles.|Qué pena que hayan quitado puestos para la función.
El acceso a la sala se habilitará diez minutos antes de la presentación.|Te dejan pasar a la sala diez minutos antes de que empiece.
Podrá consultar las letras traducidas al final del folleto.|Si quieres ver qué dicen las canciones, al final del folleto está la traducción.
Si requiere un programa impreso, puede solicitarlo en recepción.|Si quieres el programa en papel, pídelo en recepción.
La lectura de poesía se trasladó a la sala de usos múltiples.|Lo de leer poemas ahora es en el salón grande de usos múltiples.
Le agradecemos su contribución a la conservación del archivo.|Gracias por ayudarnos a cuidar todo lo que hay guardado en el archivo.
""",
    "Tecnología": """
Se recomienda actualizar el navegador antes de acceder al formulario.|Actualiza el navegador primero y de ahí abres el formulario.
La copia local se conservará aunque se elimine el archivo compartido.|Aunque borres el archivo que compartiste, te queda el de tu compu.
La cuenta admite un segundo método de verificación.|Puedes ponerle otra forma de comprobar que sí eres tú.
La aplicación solicitará acceso a la cámara en el primer uso.|La primera vez la app te va a pedir permiso para usar la cámara.
¿Podría verificar si la sincronización concluyó?|¿Me miras si ya se terminó de sincronizar?
La versión publicada corrige un problema de visualización en pantallas pequeñas.|Con la versión nueva ya se ve bien en pantallas chiquitas.
Le informo que la migración de datos concluyó sin incidencias.|Te cuento que ya pasamos los datos y todo salió bien.
Los cambios de configuración tendrán efecto tras el reinicio.|Apaga y prende otra vez para que agarren los cambios.
El sistema no admite nombres de archivo con determinados símbolos.|Si le pones ciertos símbolos al nombre, no te deja guardarlo.
Solicito acceso de lectura al directorio compartido.|¿Me das permiso para ver lo que hay en la carpeta compartida?
La conexión se restableció después de actualizar el controlador.|Actualicé el controlador y con eso volvió la conexión.|Ambas usan los mismos sustantivos técnicos. La frase informal presenta al hablante y usa «con eso volvió». La formal enfatiza el restablecimiento de la conexión.
Para recuperar la contraseña, compruebe su bandeja de correo no deseado.|Si no ves el correo de la clave, revisa en spam.
Puede prescindir del cable si activa la conexión inalámbrica.|Si prendes el wifi, ya puedes dejar de usar el cable.
La capacidad restante es insuficiente para completar la instalación.|Ya casi no tienes espacio, por eso no entra todo lo que quieres instalar.
¿Estaría disponible para verificar el funcionamiento del micrófono?|¿Tienes un minuto para ver si el micro sí funciona?
El equipo reconoció automáticamente el dispositivo conectado.|La compu detectó solita lo que le enchufaste.
El informe técnico describe las causas de la falla de autenticación.|En el informe cuentan por qué fallaba el inicio de sesión.
El envío de notificaciones se puede desactivar en las preferencias.|Si te molestan los avisos, los puedes apagar en los ajustes.
Los datos pueden exportarse antes de cancelar la suscripción.|Antes de darte de baja, saca una copia de tus datos.
La plantilla conserva los estilos aplicados a los encabezados.|Con esa plantilla te quedan igualitos los títulos.
El sonido se interrumpe al cambiar entre perfiles de salida.|Se corta el audio cada vez que cambias por dónde sale.
La opción de búsqueda permite limitar los resultados por fecha.|Puedes decirle al buscador de qué fechas quieres los resultados.
El enlace de recuperación permanecerá activo durante veinte minutos.|Tienes veinte minutos para usar el enlace y recuperar la cuenta.
Por favor, compruebe si el archivo mantiene su extensión original.|Fíjate si el archivo sigue terminando en lo mismo, porfa.
Agradezco que haya identificado el origen del fallo.|Gracias por cachar de dónde venía el problema.
La descarga quedó interrumpida tras la pérdida de conectividad.|Se fue el internet y quedó a medias la descarga.
Se han revocado los permisos asociados a la cuenta anterior.|A la cuenta vieja ya le quitaron los permisos.
El sistema permite agrupar las alertas por nivel de prioridad.|Puedes juntar los avisos según qué tan urgentes sean.
Habilitaremos la opción cuando concluya la prueba de estabilidad.|Prendemos esa opción apenas acabemos de ver si aguanta bien.
El dispositivo no conserva los ajustes después de un reinicio de fábrica.|Si lo dejas como vino de fábrica, se te borran los ajustes.
El mensaje de error aparece al adjuntar archivos comprimidos.|Sale ese error cuando le metes un archivo comprimido.
¿Podría reducir la resolución antes de subir la imagen?|¿Le bajas la resolución a la foto antes de subirla?
El historial local permite recuperar las modificaciones recientes.|En el historial de la compu puedes volver a los cambios de hace un rato.
Adjunto una captura de la pantalla donde se manifiesta el inconveniente.|Te mando una captura de donde se ve el problema.
El documento compartido admite edición simultánea.|Podemos escribir los dos a la vez en el mismo archivo.|El mismo hecho se formula como una capacidad del documento o como algo que pueden hacer dos interlocutores. «Edición simultánea» concentra el registro técnico.
La conexión se mantiene estable si se usa el puerto alternativo.|Con el otro puerto sí se queda conectada sin cortarse.
La aplicación almacena temporalmente los cambios sin conexión.|Aunque se te vaya el internet, la app te guarda lo que cambiaste por un rato.
El teclado requiere una configuración adicional para ingresar tildes.|Toca cambiar algo en el teclado para poder poner las tildes.
Si desea cambiar de cuenta, cierre la sesión actual.|Sal de la cuenta que tienes abierta si quieres entrar con otra.
El visor permite comparar dos versiones del mismo documento.|Ahí puedes poner las dos versiones del archivo lado a lado.
""",
    "Comunidad": """
La administración del barrio recibirá propuestas para el uso de la cancha.|Si se les ocurre qué hacer con la cancha, cuéntenselo a los del barrio.
Les agradeceríamos que conservaran libre el acceso a los hidrantes.|Dejen libre el espacio de los hidrantes, porfa.
El encuentro vecinal tendrá lugar en la casa comunal.|Nos juntamos los vecinos en la casa comunal.
Se solicita retirar las bicicletas antes de comenzar la limpieza.|Ojo, saquen las bicis antes de que empiecen a limpiar.
Propongo reservar una parte del presupuesto para reparar los juegos infantiles.|Yo guardaría algo de plata para arreglar los juegos de los guaguas.
Las personas interesadas pueden inscribirse al término de la asamblea.|Si alguien se quiere apuntar, que lo haga cuando acabe la reunión.
El informe recoge los acuerdos alcanzados por las delegaciones.|En el informe pusieron en qué quedaron los delegados.
Le agradecería que trasladara el aviso a los residentes del tercer piso.|¿Les pasas el aviso a los del tercer piso?
Con motivo de la feria, se cerrará el paso por la calle central.|Van a cerrar la calle del medio por lo de la feria.
Podemos facilitarle una silla si desea permanecer en la reunión.|Si te quieres quedar en la reunión, te conseguimos una silla.
Le agradezco que haya esperado a que terminara mi intervención.|Gracias por esperar a que acabara de hablar.
El servicio de recolección se trasladó a la jornada del jueves.|La basura la recogen ahora el jueves.
La convocatoria incluye a propietarios y arrendatarios.|Pueden venir los dueños y también los que arriendan.
El comité dispone de fondos para renovar el alumbrado del pasaje.|El comité sí tiene plata para cambiar las luces del callejón.
Sugiero que retomemos el asunto una vez que estén los representantes.|Yo digo que volvamos a hablar de eso cuando lleguen los representantes.
Los formularios estarán a disposición del público en el centro comunitario.|En el centro comunitario puedes agarrar uno de los formularios.
¿Podría indicar dónde se recibió la última donación?|¿Dónde dejaron lo último que donaron?
El reglamento permite el uso del salón previa reserva.|Si apartas el salón antes, sí lo puedes usar.
Le confirmo que la minga continuará aunque llueva.|Te aviso que la minga va igual aunque llueva.
Agradecemos el respaldo que nos brindaron durante la colecta.|Gracias por estar ahí y ayudar con la colecta.
Las observaciones al acta podrán presentarse hasta mañana.|Si algo del acta no te cuadra, todavía puedes decirlo mañana.
Se acordó mantener el acceso abierto hasta las veinte horas.|Quedamos en dejar abierta la entrada hasta las ocho de la noche.
La comisión publicará el destino de los fondos recaudados.|La comisión va a contar en qué se usará lo que juntamos.
Si lo considera pertinente, puede incorporar su observación al acta.|Si crees que vale la pena, mete lo que dijiste en el acta.
¿Sería posible dejar una copia del aviso en el vestíbulo?|¿Dejas una copia del aviso donde entra la gente?
El contacto del responsable figura en el reverso del volante.|Atrás del volante está el número de quien se encarga.
Se agradece la devolución de los materiales prestados al cierre de la jornada.|Antes de irse devuelvan lo que les prestamos, gracias.
Le informo que la reparación de la acera continúa pendiente.|La vereda sigue sin arreglarse, por si preguntabas.
El acceso por el patio se mantiene disponible.|Por el patio todavía se puede pasar.|El aviso formal es breve y omite tratamiento de cortesía. La variante informal anticipa el lugar y usa «todavía» con ritmo conversacional. Ambas incluyen una construcción con «se».
Agradecemos que hayan venido a conversar sobre la seguridad del sector.|Gracias por venir a hablar de cómo está la seguridad por aquí.|Ambas frases agradecen y comparten el tema. «Por aquí» y la formulación de pregunta indirecta en la segunda dan cercanía.
La junta se reunirá el viernes por la tarde.|La junta se va a reunir el viernes en la tarde.|Comparten casi todos los sustantivos. El futuro simple y la perífrasis «se va a reunir» dan un contraste tenue.
¿Me permite consultar el registro de asistentes?|¿Me deja ver la lista de los que vinieron?|Ambas pueden dirigirse a una persona de usted. El contraste está en «consultar el registro» frente a «ver la lista de los que vinieron».
Quisiera que revisáramos las cuentas antes de aprobar el gasto.|Quiero que revisemos bien las cuentas antes de aprobar el gasto.|El segmento final coincide. «Quisiera» y el imperfecto de subjuntivo atenúan la petición formal. «Bien» aporta énfasis conversacional.
Solo resta confirmar el lugar de la próxima reunión.|Solo queda confirmar dónde va a ser la próxima reunión.|Las dos son impersonales y sin jerga. «Resta» y el sintagma «el lugar» son las pistas más claras de la versión formal.
El plazo para postular termina este viernes.|Hasta este viernes hay chance de postular.|Las dos mantienen «postular» y «viernes». «Hay chance» es el giro coloquial, pese al vocabulario de convocatoria.
Si le parece, podemos empezar con las preguntas del público.|Si les parece, arranquemos con las preguntas del público.|«Si le parece» y «si les parece» se parecen y expresan cortesía. «Arranquemos» cambia el registro de la segunda.
Le agradezco que me acompañe a la oficina de la junta.|Gracias por venir conmigo a la oficina de la junta.|Comparten una acción y un lugar. El pronombre «le» y el subjuntivo «acompañe» aportan un trato más protocolario.
De acuerdo, conservaré el comprobante de la contribución.|De acuerdo, guardo el comprobante del aporte.|«De acuerdo» sirve en ambos registros. La segunda formula un compromiso con presente y vocabulario más cotidiano.
La actividad continuará si mejora el clima.|Seguimos con la actividad si mejora el clima.|La condición es idéntica. La primera usa futuro y sujeto impersonal respecto al hablante, la otra incorpora al grupo con «seguimos».
Los puestos se asignarán por orden de llegada.|Van dando los puestos según vaya llegando la gente.|Ambas explican una regla de acceso. «Le toca» y «puesto» son pistas conversacionales, aunque aparezca el pronombre «le».
""",
    "Deporte": """
El árbitro dio por concluido el encuentro tras el segundo tiempo.|El árbitro pitó el final apenas acabó el segundo tiempo.|Las dos mencionan al árbitro y al segundo tiempo. «Dio por concluido el encuentro» contrasta con el verbo deportivo cotidiano «pitó».
La sesión de entrenamiento se realizará en la cancha cubierta.|Hoy entrenamos bajo techo.
Se recomienda llegar quince minutos antes del calentamiento.|Mejor llega un cuarto de hora antes de que empecemos a calentar.
La inscripción del equipo requiere los datos de todos los integrantes.|Para apuntar al equipo, pasa los datos de todos los que juegan.
¿Podría confirmarme la disponibilidad de balones para la práctica?|¿Sabes si hay pelotas para entrenar?
Los resultados se publicarán al cierre de la jornada.|Los resultados los suben cuando acaben los partidos del día.
Agradezco que me haya reservado un lugar en el equipo.|Gracias por guardarme un puesto en el equipo.
Se suspende la actividad por el mal estado del terreno.|No jugamos porque la cancha está hecha un desastre.
El encuentro se disputará en dos períodos de veinte minutos.|Vamos a jugar dos tiempos de veinte minutos.
Puede solicitar el material en la oficina de la coordinación deportiva.|Si te hace falta algo, pídelo en la oficina de deportes.
El cambio de horario afecta únicamente a la categoría juvenil.|Le movieron la hora solo a los del juvenil.
Quisiera consultar si se permite el ingreso con alimentos.|¿Sabes si nos dejan meter comida?
La organización entregará los dorsales antes de la salida.|Antes de arrancar te dan el numerito para la carrera.
Su nombre figura en la lista de suplentes.|Sales en la lista de los que van a esperar en la banca.
La ruta de la carrera incluye un tramo de pendiente pronunciada.|En la carrera hay una subida bien brava.
El equipo local obtuvo la ventaja a falta de cinco minutos.|Los de casa se pusieron arriba cuando faltaban cinco minutos.
Conviene revisar el ajuste de las correas antes de iniciar el descenso.|Fíjate que las correas queden bien puestas antes de bajar.
El ingreso al vestuario estará restringido al personal autorizado.|Al camerino no entra cualquiera, solo los que tienen permiso.
Le agradecería que llevara el botiquín a la zona de encuentro.|¿Nos llevas el botiquín a donde nos vamos a reunir?
El calentamiento se ajustará a la intensidad de la sesión.|Vamos a calentar según qué tan fuerte sea el entreno.
Quienes completen el circuito recibirán una constancia de participación.|A los que acaben la vuelta les dan un papel que dice que participaron.
¿Podría indicarnos qué equipos continúan en competencia?|¿Me dices qué equipos siguen jugando en el torneo?
Las personas inscritas deben informar si tienen previsto retirarse de la prueba.|Si te inscribiste y te vas a salir de la prueba, avisa.
La ruta corta se mantiene habilitada para principiantes.|Si recién estás empezando, puedes tomar el camino corto.
La puntuación final se calculará a partir de tres intentos.|Van a contar los tres intentos para sacar tu puntaje.
El empate se resolverá mediante una serie de penaltis.|Si quedan iguales, se van a penales.
El acceso a las gradas permanecerá cerrado durante la revisión de seguridad.|No puedes subir a las gradas mientras hacen el chequeo de seguridad.
El organizador confirmó la participación del equipo visitante.|El que organiza dijo que sí vienen los visitantes.
Se recomienda practicar el movimiento con una carga reducida.|Haz primero el movimiento con poquito peso.
Por favor, vuelva a colocar el equipo de protección en el estante.|Deja de nuevo la protección en la repisa, por favor.|La cortesía es idéntica. El trato «vuelva» y la expresión «equipo de protección» contrastan con «deja» y la formulación más corta.
La cancha principal seguirá disponible durante el receso.|La cancha principal sigue libre en el descanso.|Ambas se parecen a un aviso. «Durante el receso» y «seguirá disponible» cargan la diferencia de registro.
¿Podemos ajustar el ritmo en la segunda vuelta?|¿Le bajamos un poco al ritmo en la segunda vuelta?|Ambas son preguntas de equipo. «Le bajamos» es un giro coloquial, aunque compartan toda la segunda mitad.
El partido se reanudó tras la revisión de la jugada.|El partido siguió después de que revisaron la jugada.|La primera condensa la secuencia en «tras la revisión». La segunda usa una subordinada cotidiana con agente implícito.
Le agradezco que me haya prestado el casco.|Gracias por prestarme el casco.|Las dos son cortas, corteses y comparten el sustantivo. La formalidad proviene de «le agradezco» y del subjuntivo compuesto.
El árbitro solicitó una revisión antes de conceder el punto.|El árbitro pidió revisar la jugada antes de dar el punto.|Ambas tienen un tono descriptivo. «Solicitó una revisión» y «conceder» contrastan con verbos de uso más corriente.
Si están de acuerdo, podemos iniciar con un ejercicio de movilidad.|Si les parece, arrancamos con algo de movilidad.|Las dos buscan acuerdo grupal. «Arrancamos» y «algo de» introducen una voz más conversacional.
El equipo volvió a entrenar tras dos días de descanso.|El equipo ya volvió a entrenar después de dos días de descanso.|La segunda conserva casi toda la oración. «Ya» y «después de» son indicios leves del registro informal.
Las llaves del gimnasio se entregarán a quien concluya el turno.|Las llaves del gimnasio se las dan al que cierre el turno.|«Se» aparece en ambas. La duplicación «se las» y «al que cierre» hacen menos institucional la segunda.
El torneo no continuará si persiste la lluvia.|El torneo no sigue si no para la lluvia.|La estructura y el tema se mantienen. «Persiste» y el futuro formal contrastan con presentes y negación conversacional.
Le confirmaré mañana si contamos con una cancha adicional.|Mañana te confirmo si tenemos otra cancha.|El par comparte verbo, fecha y tema. «Le» y «contamos con» contrastan con «te» y el presente con valor de futuro.
""",
    "Proyectos": """
El borrador incorpora las observaciones recibidas durante la primera revisión.|Al borrador ya le metimos lo que nos dijeron la primera vez.
Solicito una copia del presupuesto para incorporar los valores al informe.|¿Me pasas el presupuesto para poner esos valores en el informe?
La estimación contempla los recursos disponibles hasta diciembre.|El cálculo va con lo que tenemos de aquí a diciembre.
El equipo resolvió aplazar la prueba piloto una semana.|Quedamos en correr la prueba piloto una semana.
Se requiere su conformidad para publicar la versión final.|Hace falta que nos digas que sí para subir la versión final.
Le envío los resultados preliminares para su revisión.|Te paso lo que salió hasta ahora para que lo mires.
Quisiera precisar qué criterios se utilizaron para seleccionar la muestra.|¿Me cuentas en qué se fijaron para escoger la muestra?
La versión anterior se conservará como referencia.|Dejemos guardada la versión vieja para acordarnos de cómo estaba.
El documento no incluye aún el costo de mantenimiento.|Todavía falta meter lo que cuesta mantenerlo.|Las dos señalan un costo omitido. «No incluye aún» adopta el tono de un informe. «Falta meter» describe la misma omisión desde una conversación de trabajo.
La exposición comenzará con un resumen del problema identificado.|Al comienzo vamos a contar por encima cuál fue el problema que vimos.
Considero pertinente incluir ejemplos representativos de cada categoría.|Yo pondría un par de ejemplos que se entiendan de cada grupo.
El enlace se compartirá cuando se habiliten los permisos de acceso.|Apenas se pueda abrir, les pasamos el enlace.
La coordinación solicitó que cada responsable verificara sus entregables.|Desde coordinación pidieron que cada uno revise lo que le toca entregar.
El segundo prototipo incorpora un soporte de mayor tamaño.|Al segundo prototipo le pusimos un soporte más grandecito.
Agradezco que haya precisado el alcance de su propuesta.|Gracias por explicar mejor hasta dónde llega tu idea.
¿Podría añadir la fecha de la última modificación?|¿Me pones de cuándo es el último cambio?
El archivo compartido contiene un registro de las modificaciones.|En el archivo compartido puedes ver qué se le cambió.
Conviene reproducir el cálculo con los valores originales.|Mejor saca la cuenta otra vez con los números del principio.
La demostración quedará a cargo de la persona responsable del diseño.|La demo la va a hacer quien armó el diseño.
El instrumento se aplicará a un grupo de participantes voluntarios.|Vamos a probar el formulario con gente que quiera ayudar.
Se propone ajustar el título para reflejar el alcance del proyecto.|Cambiémosle el título un poco para que diga bien de qué va el proyecto.
Le ruego mantener la numeración original de las tablas.|Porfa, deja las tablas con los mismos números.
La gráfica presenta una variación pronunciada durante el último período.|En la última parte la línea pega un cambio fuerte.
La estimación preliminar coincide con los valores de referencia.|Por ahora nos salen números parecidos a los que teníamos de referencia.
El informe distingue los gastos previstos de los ejecutados.|En el informe se ve qué pensábamos gastar y qué gastamos de verdad.
Podrá consultar las fuentes de datos en el anexo metodológico.|Si quieres ver de dónde sacamos los datos, mira el anexo de metodología.
La propuesta requiere un apartado adicional sobre el cronograma.|A la propuesta le falta una parte para decir cuándo hacemos cada cosa.
Se descartaron las observaciones que no cumplían los criterios establecidos.|Sacamos los datos que no pasaban las reglas que pusimos.
Le agradecería que verificara la fecha en el archivo compartido.|¿Me revisas la fecha que quedó en el archivo compartido?
El total coincide con la cifra registrada en el informe.|El total da lo mismo que pusimos en el informe.|Ambas son declaraciones breves sobre el mismo cálculo. «Coincide con la cifra registrada» es más elaborado que «da lo mismo que pusimos».
Considero que la primera opción responde mejor al objetivo.|Yo creo que la primera opción encaja mejor con lo que buscamos.|Comparten una valoración. El uso explícito de «yo creo» y «lo que buscamos» marca cercanía frente a «responde al objetivo».
Las primeras observaciones permiten identificar dos problemas recurrentes.|En lo primero que vimos salen dos problemas que se repiten.|Ambas usan casi el mismo contenido técnico. El sintagma «problemas recurrentes» comprime lo que la segunda dice en una relativa.
Revisaremos la redacción una vez que termine la recopilación de datos.|Cuando acabemos de juntar los datos, revisamos cómo quedó escrito.|Las dos usan primera persona plural. La diferencia viene de «recopilación» y «una vez que» frente a «juntar» y el orden conversacional.
La etapa de revisión puede tomar una semana adicional.|Capaz nos toma otra semana revisar todo.|La duración no distingue el registro. El modal coloquial «capaz» y «nos» sitúan la frase en una conversación de equipo.
Los resultados se mantienen al incorporar las últimas observaciones.|Con los últimos datos los resultados quedan igual.|Ambas expresan estabilidad y son breves. «Al incorporar» y «se mantienen» marcan el registro técnico.
¿Tiene alguna observación sobre el último cambio?|¿Algo que decir sobre el último cambio?|Las dos son preguntas cortas y comparten el final. La omisión de verbo en la segunda suena a pregunta rápida.
Gracias por revisar el cuadro antes de remitirlo.|Gracias por mirar el cuadro antes de mandarlo.|El agradecimiento y casi toda la estructura coinciden. «Revisar/remitir» frente a «mirar/mandar» concentra la diferencia.
El proyecto requiere que definamos quién realizará cada tarea.|Nos falta ponernos de acuerdo en quién hace cada cosa.|Comparten el acto de asignar tareas. «Requiere que definamos» condensa la instrucción y «cada cosa» la vuelve más coloquial.
Si no surge otra observación, daremos por cerrada la revisión.|Si nadie tiene algo más que decir, cerramos la revisión.|La palabra «revisión» y la condición son comunes. «Daremos por cerrada» tiene una fórmula formal frente al directo «cerramos».
Los dos gráficos presentan la misma distribución.|En los dos gráficos los datos se reparten igual.|Las frases son descriptivas y sin trato. «Presentan la misma distribución» se apoya en vocabulario estadístico y la segunda lo parafrasea.
""",
}

In [4]:
filas = []
for tema, bloque in CORPUS.items():
    for numero, linea in enumerate(bloque.strip().splitlines(), start=1):
        formal, informal, *nota = linea.split("|")
        assert len(nota) <= 1
        for etiqueta, texto in enumerate([formal, informal]):
            filas.append({
                "id": f"{tema}-{numero:02d}-{etiqueta}",
                "grupo": f"{tema}-{numero:02d}",
                "tema": tema,
                "texto": texto.strip(),
                "label": etiqueta,
                "estilo": ID_A_ESTILO[etiqueta],
                "nota_registro": nota[0].strip() if nota else "",
                "contraste_sutil": bool(nota),
            })

datos = pd.DataFrame(filas)
normalizadas = datos["texto"].map(
    lambda texto: " ".join(re.findall(r"\w+", unicodedata.normalize("NFC", texto).casefold()))
)
assert len(datos) == 800 and datos["id"].is_unique
assert normalizadas.str.strip().ne("").all() and normalizadas.is_unique
assert datos.groupby("grupo")["label"].apply(set).eq(set(IDS)).all()

balance_temas = pd.crosstab(datos["tema"], datos["estilo"]).reindex(columns=CLASES)
balance_temas.loc["Total"] = balance_temas.sum()
display(balance_temas)
print(f"Pares con contraste sutil: {datos.loc[datos['contraste_sutil'], 'grupo'].nunique()}")
display(datos.groupby("estilo")["texto"].apply(
    lambda textos: textos.str.split().str.len().median()
).rename("Mediana de palabras por frase"))
display(datos.groupby("tema", sort=False).head(2)[["tema", "estilo", "texto"]].head(10))

estilo,formal,informal
tema,,
Compras y servicios,40,40
Comunidad,40,40
Cultura,40,40
Deporte,40,40
Estudios,40,40
Hogar,40,40
Movilidad,40,40
Proyectos,40,40
Tecnología,40,40


Pares con contraste sutil: 50


,Mediana de palabras por frase
estilo,
formal,10.0
informal,10.0


,tema,estilo,texto
0,Estudios,formal,Se aceptarán las prácticas enviadas hasta las ...
1,Estudios,informal,Hasta las seis todavía puedes mandar la práctica.
80,Trabajo,formal,¿Podría actualizar el inventario antes de la v...
81,Trabajo,informal,¿Me dejas al día el inventario antes de que ve...
160,Compras y servicios,formal,El importe se acreditará en su cuenta dentro d...
161,Compras y servicios,informal,En unos cinco días laborables ya te cae la pla...
240,Hogar,formal,Le agradecería que mantuviera la ventana cerra...
241,Hogar,informal,"¿Dejas cerrada la ventana en la noche, porfa?"
320,Movilidad,formal,El próximo autobús saldrá del andén siete.
321,Movilidad,informal,"El bus que sigue sale del andén siete, pilas."


## 2. Partición por pares

En cada tema se asignan 28 pares a entrenamiento, 6 a validación y 6 a prueba. Los dos registros de una misma intención permanecen juntos. La asignación se reproduce con la semilla 42. Los casos de contraste sutil están presentes en las tres particiones. Validación y prueba reciben la misma cantidad de estos casos por tema, con una mayor proporción que entrenamiento. La dificultad se fijó al redactar el corpus, antes de entrenar.

Validación sirve para elegir la regularización de la línea base y el mejor checkpoint de BETO. Prueba se reserva para la evaluación final. Esta prueba contiene deliberadamente casos de registro cercano y comparte los diez temas de entrenamiento.

In [5]:
rng = np.random.default_rng(SEMILLA)
asignaciones = {}
for tema, bloque in datos.groupby("tema", sort=False):
    pares = bloque.drop_duplicates("grupo")
    sutiles = rng.permutation(pares.loc[pares["contraste_sutil"], "grupo"].to_numpy())
    regulares = rng.permutation(pares.loc[~pares["contraste_sutil"], "grupo"].to_numpy())

    sutiles_por_evaluacion = max(1, len(sutiles) // 3) if len(sutiles) else 0
    sutiles_val, sutiles_prueba, sutiles_ent = np.split(
        sutiles, [sutiles_por_evaluacion, 2 * sutiles_por_evaluacion]
    )
    corte_ent = 28 - len(sutiles_ent)
    corte_val = corte_ent + 6 - len(sutiles_val)
    grupos_ent = np.concatenate([sutiles_ent, regulares[:corte_ent]])
    grupos_val = np.concatenate([sutiles_val, regulares[corte_ent:corte_val]])
    grupos_prueba = np.concatenate([sutiles_prueba, regulares[corte_val:]])
    for nombre, grupos in zip(PARTICIONES, [grupos_ent, grupos_val, grupos_prueba]):
        for grupo in grupos:
            assert grupo not in asignaciones
            asignaciones[grupo] = nombre

datos["partición"] = datos["grupo"].map(asignaciones)
assert datos["partición"].notna().all()
assert datos.groupby("grupo")["partición"].nunique().eq(1).all()

partes = {
    nombre: datos.loc[datos["partición"].eq(nombre)].sample(frac=1, random_state=SEMILLA).reset_index(drop=True)
    for nombre in PARTICIONES
}
entrenamiento, validacion, prueba = (partes[nombre] for nombre in PARTICIONES)

resumen = pd.crosstab(datos["partición"], datos["estilo"]).reindex(
    index=PARTICIONES, columns=CLASES
)
assert resumen.to_numpy().tolist() == [[280, 280], [60, 60], [60, 60]]
assert pd.crosstab(datos["tema"], datos["partición"]).reindex(
    columns=PARTICIONES
).eq([56, 12, 12], axis="columns").all().all()
resumen["Total"] = resumen.sum(axis=1)
resumen["Porcentaje"] = (100 * resumen["Total"] / len(datos)).map(lambda valor: f"{valor:.0f} %")
resumen["Frases de contraste sutil"] = [int(parte["contraste_sutil"].sum()) for parte in partes.values()]
display(resumen)

estilo,formal,informal,Total,Porcentaje,Frases de contraste sutil
partición,,,,,
entrenamiento,280,280,560,70 %,24
validación,60,60,120,15 %,38
prueba,60,60,120,15 %,38


## 3. Línea base · TF-IDF y regresión logística

TF-IDF pondera palabras y pares de palabras según su frecuencia en las frases de entrenamiento. La regresión logística aprende qué combinaciones de rasgos favorecen cada clase. Se conserva la información de tildes y se admiten palabras de una letra, como «a» o «y». Se compara la regularización en validación y se elige el valor de **C** con mayor F1-macro. Un empate favorece el menor C.

In [6]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

vectorizador = TfidfVectorizer(
    ngram_range=(1, 2), min_df=2, sublinear_tf=True, token_pattern=r"(?u)\b\w+\b"
)

def metricas(reales, predicciones):
    return {
        "accuracy": accuracy_score(reales, predicciones),
        "f1_macro": f1_score(
            reales, predicciones, labels=IDS, average="macro", zero_division=0
        ),
    }

inicio = perf_counter()
x_ent = vectorizador.fit_transform(entrenamiento["texto"])
x_val = vectorizador.transform(validacion["texto"])
candidatos = {}
metricas_base = []

for c in (0.5, 1.0, 5.0):
    regresion = LogisticRegression(C=c, max_iter=2000, random_state=SEMILLA)
    regresion.fit(x_ent, entrenamiento["label"])
    candidatos[c] = regresion
    metricas_base.append({"C": c, **metricas(validacion["label"], regresion.predict(x_val))})

seleccion_base = pd.DataFrame(metricas_base).sort_values(
    ["f1_macro", "C"], ascending=[False, True]
)
c_elegido = float(seleccion_base.iloc[0]["C"])
linea_base = candidatos[c_elegido]
segundos_base = perf_counter() - inicio

display(seleccion_base.round(4))
print(f"C seleccionado: {c_elegido:g} | Rasgos TF-IDF: {x_ent.shape[1]}")

,C,accuracy,f1_macro
2,5.0,0.9167,0.9165
0,0.5,0.8917,0.8915
1,1.0,0.8917,0.8915


C seleccionado: 5 | Rasgos TF-IDF: 1269


## 4. Ajuste fino de BETO

Se adapta el flujo de **Fine_tuning_emociones_DistilBERT.ipynb** de la clase. Conservamos la tokenización por lotes, el padding dinámico, el cálculo de métricas en el Trainer y la selección del mejor checkpoint por F1-macro. La tarea usa **dccuchile/bert-base-spanish-wwm-cased**, un modelo preentrenado en español, y una cabeza nueva con dos salidas. El aviso de carga identifica los pesos nuevos del clasificador y del pooler. Se ajustan todos los parámetros del modelo.

Se fija una tasa de aprendizaje de **2 × 10⁻⁵**, un máximo de **4 épocas**, lotes de **8** para entrenamiento y calentamiento durante el primer **10 %** de los pasos. Se comprueba la cobertura de la longitud máxima de 64 tokens en entrenamiento. Al final de cada época se evalúa el modelo en validación. La época con mejor F1-macro determina los pesos para la evaluación final.

In [ ]:
from tempfile import TemporaryDirectory

import torch
from datasets import Dataset
from transformers import (
    AutoModelForSequenceClassification, AutoTokenizer, DataCollatorWithPadding,
    Trainer, TrainingArguments, set_seed,
)

MODELO_BASE = "dccuchile/bert-base-spanish-wwm-cased"
MAX_TOKENS = 64
usa_cuda = torch.cuda.is_available()
usa_bf16 = usa_cuda and torch.cuda.is_bf16_supported(including_emulation=False)
usa_fp16 = usa_cuda and not usa_bf16 and torch.cuda.get_device_capability(0)[0] >= 7

set_seed(SEMILLA)
tokenizer = AutoTokenizer.from_pretrained(MODELO_BASE)

longitudes = np.array([
    len(ids) for ids in tokenizer(entrenamiento["texto"].tolist(), verbose=False)["input_ids"]
])
print(f"Tokens en entrenamiento · mediana {np.median(longitudes):.0f} | máximo {longitudes.max()}")
print(f"Frases de entrenamiento truncadas a {MAX_TOKENS} tokens: {(longitudes > MAX_TOKENS).sum()}")

def tokenizar(lote):
    return tokenizer(lote["texto"], truncation=True, max_length=MAX_TOKENS)

def preparar_dataset(parte):
    datos_hf = Dataset.from_pandas(parte[["texto", "label"]], preserve_index=False)
    return datos_hf.map(tokenizar, batched=True, remove_columns=["texto"])

train_tok = preparar_dataset(entrenamiento)
val_tok = preparar_dataset(validacion)
collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [ ]:
def calcular_metricas(resultado):
    return metricas(resultado.label_ids, np.argmax(resultado.predictions, axis=-1))

carpeta_checkpoints = TemporaryDirectory(prefix="beto-estilo-")
set_seed(SEMILLA)
modelo = AutoModelForSequenceClassification.from_pretrained(
    MODELO_BASE,
    num_labels=len(CLASES),
    id2label=ID_A_ESTILO,
    label2id=ESTILO_A_ID,
)
argumentos = TrainingArguments(
    output_dir=carpeta_checkpoints.name,
    learning_rate=2e-5,
    num_train_epochs=4,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    weight_decay=0.01,
    warmup_steps=0.10,
    optim="adamw_torch",
    logging_strategy="epoch",
    eval_strategy="epoch",
    save_strategy="epoch",
    save_only_model=True,
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    bf16=usa_bf16,
    fp16=usa_fp16,
    report_to="none",
    seed=SEMILLA,
    data_seed=SEMILLA,
)

trainer = Trainer(
    model=modelo, args=argumentos, train_dataset=train_tok, eval_dataset=val_tok,
    data_collator=collator, processing_class=tokenizer, compute_metrics=calcular_metricas,
)

print("Dispositivo:", argumentos.device)
if usa_cuda:
    print("GPU:", torch.cuda.get_device_name(0))
print("Precisión:", "BF16" if usa_bf16 else "FP16" if usa_fp16 else "FP32")
print("Parámetros entrenables:", f"{sum(p.numel() for p in modelo.parameters() if p.requires_grad):,}")

In [ ]:
# Esta celda ejecuta el ajuste fino.
inicio = perf_counter()
trainer.train()
segundos_beto = perf_counter() - inicio

print(f"Mejor F1-macro de validación: {trainer.state.best_metric:.4f}")
print(f"Duración del ajuste y evaluación por época: {segundos_beto / 60:.2f} min")

## 5. Curvas de aprendizaje

La pérdida de entrenamiento resume los lotes de cada época. La pérdida de validación se mide al final de la época con el modelo resultante. La línea vertical indica el checkpoint que se recupera para prueba. F1-macro da el mismo peso al F1 de ambos registros.

In [ ]:
historial = pd.DataFrame(trainer.state.log_history)
curvas = historial.dropna(subset=["eval_loss"])[
    ["epoch", "step", "eval_loss", "eval_accuracy", "eval_f1_macro"]
].merge(
    historial.dropna(subset=["loss"])[["step", "loss"]], on="step", validate="one_to_one"
).sort_values("epoch").reset_index(drop=True)

paso_elegido = int(trainer.state.best_model_checkpoint.rsplit("-", 1)[-1])
mejor_epoca = curvas.loc[curvas["step"].eq(paso_elegido)].iloc[0]
primera, ultima = curvas.iloc[0], curvas.iloc[-1]
epoca_elegida = int(mejor_epoca["epoch"])

fig, ejes = plt.subplots(1, 2, figsize=(11, 4), layout="constrained")
ejes[0].plot(curvas["epoch"], curvas["loss"], "o-", label="Entrenamiento")
ejes[0].plot(curvas["epoch"], curvas["eval_loss"], "s-", label="Validación")
ejes[0].set(title="Pérdida por época", ylabel="Entropía cruzada")
ejes[1].plot(curvas["epoch"], curvas["eval_accuracy"], "o-", label="Accuracy")
ejes[1].plot(curvas["epoch"], curvas["eval_f1_macro"], "s--", label="F1-macro")
ejes[1].set(title="Desempeño en validación", ylabel="Puntuación", ylim=(0, 1.03))

for eje in ejes:
    eje.axvline(epoca_elegida, color="gray", linestyle=":", label="Checkpoint elegido")
    eje.set(xlabel="Época", xticks=curvas["epoch"])
    eje.grid(alpha=0.2)
    eje.legend()

plt.show()
display(curvas.drop(columns="step").round(4))

if ultima["loss"] < primera["loss"] and ultima["eval_loss"] > primera["eval_loss"]:
    lectura_curvas = "La pérdida de entrenamiento bajó mientras la de validación aumentó entre la primera y la última época. Esto es compatible con sobreajuste."
elif ultima["eval_loss"] < primera["eval_loss"]:
    lectura_curvas = "La pérdida de validación terminó por debajo de la primera época. En ese intervalo mejoró la calidad de las probabilidades según la entropía cruzada."
else:
    lectura_curvas = "La pérdida de validación terminó igual o por encima de la primera época. Conviene leerla junto con el F1-macro de cada época."

display(Markdown(
    f"El F1-macro en validación pasó de **{primera['eval_f1_macro']:.3f}** a "
    f"**{ultima['eval_f1_macro']:.3f}** entre la primera y la última época. "
    f"Se recuperó la época **{epoca_elegida}** con **{mejor_epoca['eval_f1_macro']:.3f}**. "
    + lectura_curvas
))

## 6. Evaluación y comparación

Se calculan accuracy y F1-macro para ambos modelos en las mismas frases de validación y de prueba. El **classification_report** desglosa precisión, recall, F1 y número de ejemplos por clase. Las matrices de prueba muestran etiquetas reales en las filas y predicciones en las columnas. Los valores fuera de la diagonal indican la dirección de cada confusión.

In [ ]:
predicciones = {}
for nombre, parte in (("validación", validacion), ("prueba", prueba)):
    entrada = val_tok if nombre == "validación" else preparar_dataset(parte)
    salida = trainer.predict(entrada, metric_key_prefix="pred")
    assert np.array_equal(salida.label_ids, parte["label"].to_numpy())

    probabilidades_beto = torch.softmax(torch.as_tensor(salida.predictions), dim=1).numpy()
    probabilidades_base = linea_base.predict_proba(vectorizador.transform(parte["texto"]))
    assert np.array_equal(linea_base.classes_, IDS)

    for modelo_nombre, probabilidades in (
        ("BETO ajustado", probabilidades_beto),
        ("TF-IDF + regresión logística", probabilidades_base),
    ):
        tabla = parte.copy()
        tabla["predicción"] = probabilidades.argmax(axis=1)
        tabla["confianza"] = probabilidades.max(axis=1)
        predicciones[nombre, modelo_nombre] = tabla

comparacion = pd.DataFrame([
    {"Partición": particion, "Modelo": modelo_nombre, **metricas(tabla["label"], tabla["predicción"])}
    for (particion, modelo_nombre), tabla in predicciones.items()
]).rename(columns={"accuracy": "Accuracy", "f1_macro": "F1-macro"})
display(comparacion.round(4))

beto_test = predicciones["prueba", "BETO ajustado"]
base_test = predicciones["prueba", "TF-IDF + regresión logística"]
matrices = {}

fig, ejes = plt.subplots(1, 2, figsize=(10, 4), layout="constrained")
for eje, (nombre, tabla) in zip(ejes, [
    ("BETO ajustado", beto_test), ("TF-IDF + regresión logística", base_test)
]):
    print(f"\n{nombre} · conjunto de prueba")
    print(classification_report(
        tabla["label"], tabla["predicción"], labels=IDS, target_names=CLASES,
        digits=3, zero_division=0,
    ))
    matriz = confusion_matrix(tabla["label"], tabla["predicción"], labels=IDS)
    matrices[nombre] = matriz
    ConfusionMatrixDisplay(matriz, display_labels=CLASES).plot(
        cmap="Blues", values_format="d", ax=eje, colorbar=False,
        im_kw={"vmin": 0, "vmax": 60},
    )
    eje.set(title=nombre, xlabel="Predicción", ylabel="Etiqueta real")

plt.show()

contrastes = []
for nombre, tabla in [("BETO ajustado", beto_test), ("TF-IDF + regresión logística", base_test)]:
    for es_sutil, segmento in tabla.groupby("contraste_sutil"):
        contrastes.append({
            "Modelo": nombre,
            "Contraste": "Sutil" if es_sutil else "Regular",
            "N": len(segmento),
            "Errores": int(segmento["predicción"].ne(segmento["label"]).sum()),
            "F1-macro": metricas(segmento["label"], segmento["predicción"])["f1_macro"],
        })

display(pd.DataFrame(contrastes).round(4))

## 7. Análisis cualitativo de hasta diez errores observados

Se empieza con los errores de BETO en prueba y se completa con los de BETO en validación si hace falta. Si aún faltan casos, se revisan errores distintos de la línea base, con prioridad para prueba. Dentro de cada origen, el orden va de mayor a menor probabilidad asignada a la clase incorrecta. Cada frase se comenta una vez.

En cada caso se lee la frase junto a la versión de registro opuesto. Las notas describen evidencia textual y el comportamiento observado en el par. Son hipótesis sobre el error. La confianza es la probabilidad estimada por el modelo. Si aparecen menos de diez errores distintos, se reporta cuántos hubo.

In [ ]:
# Opcional. Después de ver la tabla, añada observaciones propias por identificador.
# Los comentarios se conservan al reiniciar y ejecutar el notebook.
COMENTARIOS_EQUIPO = {
    # "Comunidad-29-0": "Escriba aquí la observación concreta del equipo.",
}

In [ ]:
PATRONES_REGISTRO = {
    "cortesía compartida": r"\b(?:por favor|gracias|agradezco|agradecemos|perdón|disculpas)\b",
    "trato cercano": r"\b(?:tú|tu|tus|te|tienes|puedes|quieres|vas a|dime|pásame)\b",
    "fórmula institucional": r"\b(?:se solicita|se recomienda|se requiere|se ruega|se adjunta|deberá\w*|le agradecer\w*|podría|solicito|conforme a)\b",
    "giro coloquial": r"\b(?:porfa|pilas|de una|qué onda|chance|toca|ojo|recién|ahorita|compu|profe|expo|peli|bulla|refri|carro|ratito|poquito)\b",
}

def analizar_error(caso, par):
    if caso["nota_registro"]:
        patron = "contraste sutil"
        indicio = caso["nota_registro"]
    else:
        coincidencias = {
            nombre: re.search(patron, caso["texto"], flags=re.IGNORECASE)
            for nombre, patron in PATRONES_REGISTRO.items()
        }
        cortesia = coincidencias["cortesía compartida"]
        cercano = coincidencias["trato cercano"]
        institucional = coincidencias["fórmula institucional"]
        coloquial = coincidencias["giro coloquial"]

        if cortesia and caso["label"] == ESTILO_A_ID["informal"]:
            patron = "cortesía compartida"
            indicio = (
                f"«{cortesia.group()}» expresa cortesía en un mensaje conversacional "
                "y también se usa en mensajes formales. Puede haber pesado más que el resto de la frase."
            )
        elif institucional and caso["label"] == ESTILO_A_ID["formal"]:
            patron = "fórmula institucional omitida"
            indicio = (
                f"«{institucional.group()}» aporta formalidad. La predicción es compatible con "
                "que otros rasgos de la oración hayan predominado sobre esa fórmula."
            )
        elif coloquial or cercano:
            marca = (coloquial or cercano).group()
            patron = "señal conversacional"
            indicio = (
                f"«{marca}» tiene un uso conversacional. Su peso depende de quién habla y del "
                "resto de la frase. Conviene contrastarlo con el tratamiento del otro texto del par."
            )
        else:
            palabras = re.findall(r"\b[^\W\d_]{4,}\b", caso["texto"].casefold())
            palabras_par = set(re.findall(r"\b[^\W\d_]{4,}\b", par["texto"].casefold()))
            comunes = list(dict.fromkeys(palabra for palabra in palabras if palabra in palabras_par))
            if comunes:
                patron = "vocabulario compartido"
                ejemplos = ", ".join(f"«{palabra}»" for palabra in comunes[:4])
                indicio = (
                    f"El par comparte {ejemplos}. Este vocabulario describe la situación en ambos "
                    "registros y podría dominar sobre las diferencias de redacción."
                )
            else:
                patron = "registro dependiente de la formulación"
                fragmento = " ".join(caso["texto"].split()[:5])
                indicio = (
                    f"El inicio «{fragmento}» carece de una fórmula inequívoca de trato. "
                    "La distinción depende de cómo se expresa la intención en la frase completa."
                )

    prediccion_par = ID_A_ESTILO[int(par["predicción"])]
    if int(par["predicción"]) == int(caso["predicción"]):
        conducta = f"El modelo asignó «{prediccion_par}» a las dos versiones."
    else:
        conducta = (
            f"Al cambiar la redacción, predijo «{prediccion_par}» para la versión "
            f"«{par['estilo']}». También confundió esa versión."
        )
    return patron, f"{indicio} {conducta}"

origenes = [
    ("prueba", "BETO ajustado"),
    ("validación", "BETO ajustado"),
    ("prueba", "TF-IDF + regresión logística"),
    ("validación", "TF-IDF + regresión logística"),
]

revisados = []
ids_revisados = set()
conteo_errores = Counter()

for particion, modelo_nombre in origenes:
    tabla = predicciones[particion, modelo_nombre]
    por_grupo_y_etiqueta = tabla.set_index(["grupo", "label"])
    errores = tabla.loc[tabla["predicción"].ne(tabla["label"])].sort_values(
        ["confianza", "id"], ascending=[False, True]
    )
    conteo_errores[modelo_nombre, particion] = len(errores)
    for _, caso in errores.iterrows():
        if caso["id"] in ids_revisados or len(revisados) == 10:
            continue
        par = por_grupo_y_etiqueta.loc[(caso["grupo"], 1 - int(caso["label"]))]
        patron, analisis = analizar_error(caso, par)
        revisados.append({
            "id": caso["id"],
            "modelo": modelo_nombre,
            "partición": particion,
            "tema": caso["tema"],
            "frase": caso["texto"],
            "real": caso["estilo"],
            "predicha": ID_A_ESTILO[int(caso["predicción"])],
            "confianza": float(caso["confianza"]),
            "versión del otro registro": par["texto"],
            "patrón": patron,
            "análisis": analisis,
            "comentario del equipo": COMENTARIOS_EQUIPO.get(caso["id"], ""),
        })
        ids_revisados.add(caso["id"])

tabla_conteos = pd.Series(conteo_errores, name="Errores observados").rename_axis(
    ["Modelo", "Partición"]
).unstack()
display(tabla_conteos)
print(f"Errores distintos revisados: {len(revisados)} de 10 solicitados.")

for numero, fila in enumerate(revisados, start=1):
    comentario = fila["comentario del equipo"]
    display(Markdown(
        f"**{numero}. {fila['id']}** · {fila['modelo']} · {fila['partición']}\n\n"
        f"> {fila['frase']}\n\n"
        f"Etiqueta **{fila['real']}** · Predicción **{fila['predicha']}** "
        f"· Probabilidad de la predicción **{fila['confianza']:.1%}**\n\n"
        f"**Versión del otro registro.** {fila['versión del otro registro']}\n\n"
        f"**{fila['patrón'].capitalize()}.** {fila['análisis']}"
        + (f"\n\n**Observación del equipo.** {comentario}" if comentario else "")
    ))

patrones_observados = Counter(fila["patrón"] for fila in revisados)
if patrones_observados:
    display(pd.Series(dict(patrones_observados.most_common()), name="Casos revisados").rename_axis("Patrón"))
else:
    print("Los dos modelos clasificaron correctamente todas las frases revisadas en validación y prueba.")

## 8. Conclusión

La siguiente celda integra el resultado en prueba, las curvas y los patrones de los errores observados. Las cifras corresponden a esta ejecución y a este corpus sintético.

In [ ]:
resumen_prueba = comparacion.loc[comparacion["Partición"].eq("prueba")].set_index("Modelo")
f1_beto = float(resumen_prueba.loc["BETO ajustado", "F1-macro"])
f1_base = float(resumen_prueba.loc["TF-IDF + regresión logística", "F1-macro"])
brecha = f1_beto - f1_base

if np.isclose(brecha, 0):
    diferencia = "Los dos modelos empataron en F1-macro."
elif brecha > 0:
    diferencia = f"BETO superó a TF-IDF por **{100 * brecha:.2f} puntos porcentuales** de F1-macro."
else:
    diferencia = f"TF-IDF superó a BETO por **{-100 * brecha:.2f} puntos porcentuales** de F1-macro."

matriz_beto = matrices["BETO ajustado"]
formal_informal = matriz_beto[0, 1]
informal_formal = matriz_beto[1, 0]
patrones_texto = ", ".join(
    f"{patron} ({cantidad})" for patron, cantidad in patrones_observados.most_common(2)
)
parrafo_errores = (
    f"Entre los {len(revisados)} errores revisados, los patrones más frecuentes fueron {patrones_texto}."
    if patrones_texto else "En validación y prueba no se observaron errores para el análisis cualitativo."
)
accuracy_beto = float(resumen_prueba.loc["BETO ajustado", "Accuracy"])
accuracy_base = float(resumen_prueba.loc["TF-IDF + regresión logística", "Accuracy"])

display(Markdown(
    f"En las **{len(prueba)} frases de prueba**, BETO obtuvo **{f1_beto:.3f}** en F1-macro "
    f"y **{accuracy_beto:.3f}** en accuracy. La línea base obtuvo **{f1_base:.3f}** y "
    f"**{accuracy_base:.3f}**, respectivamente. {diferencia}\n\n"
    f"Se eligió la época **{epoca_elegida}** por su F1-macro de validación "
    f"(**{mejor_epoca['eval_f1_macro']:.3f}**). {lectura_curvas} "
    f"El ajuste de BETO con validación y checkpoints tomó **{segundos_beto:.1f} s**. "
    f"La preparación y selección de la línea base tomó **{segundos_base:.2f} s** en el mismo entorno.\n\n"
    f"BETO predijo informal para **{formal_informal}** frases formales y formal para "
    f"**{informal_formal}** frases informales. {parrafo_errores} El alcance de esta comparación "
    "se limita a frases redactadas para la práctica y a una partición de diez temas conocidos. "
    "Medir la generalización a conversaciones reales requiere otro corpus."
))

## 9. Experimento por grupos · Tasa de aprendizaje y épocas

Cada fila de BETO cambia un solo hiperparámetro respecto a la referencia de la clase, que usa una tasa de **2 × 10⁻⁵** y **3 épocas**. Cada corrida parte del mismo modelo preentrenado y conserva la semilla, la partición, los lotes y el optimizador. La corrida de 4 épocas de las secciones anteriores tiene su propio historial de entrenamiento.

El **F1-macro** de esta tabla se calcula en **validación al terminar la última época** de cada corrida. Esto permite medir el efecto de entrenar durante 1, 3 o 6 épocas. La prueba queda reservada para la evaluación final de la sección 6.

Para completar la tabla, ejecute las tres celdas de esta sección en orden. La segunda ejecuta la referencia y los cinco grupos por defecto. Si en clase les asignaron un grupo, deje solo su número en esa celda, por ejemplo, `GRUPOS_A_EJECUTAR = ("3",)`.

In [ ]:
from dataclasses import replace
from gc import collect

configuraciones_experimento = [
    ("Base", "Configuración de referencia", "2e-5 · 3 épocas", 2e-5, 3),
    ("1", "Tasa de aprendizaje", "5e-5", 5e-5, 3),
    ("2", "Tasa de aprendizaje", "1e-4", 1e-4, 3),
    ("3", "Tasa de aprendizaje", "1e-3", 1e-3, 3),
    ("4", "Épocas", "1", 2e-5, 1),
    ("5", "Épocas", "6", 2e-5, 6),
]
resultados_experimento = {}


def entrenar_y_evaluar_estilo(tasa, epocas):
    with TemporaryDirectory(prefix="beto-estilo-experimento-") as carpeta:
        args_experimento = replace(
            argumentos,
            output_dir=carpeta,
            learning_rate=tasa,
            num_train_epochs=epocas,
            do_eval=False,
            eval_strategy="no",
            save_strategy="no",
            logging_strategy="no",
            load_best_model_at_end=False,
            disable_tqdm=True,
        )
        set_seed(SEMILLA)
        modelo_experimento = AutoModelForSequenceClassification.from_pretrained(
            MODELO_BASE,
            num_labels=len(CLASES),
            id2label=ID_A_ESTILO,
            label2id=ESTILO_A_ID,
        )
        entrenador_experimento = Trainer(
            model=modelo_experimento,
            args=args_experimento,
            train_dataset=train_tok,
            eval_dataset=val_tok,
            data_collator=collator,
            processing_class=tokenizer,
            compute_metrics=calcular_metricas,
        )
        entrenador_experimento.train()
        resultado = entrenador_experimento.evaluate()
        del entrenador_experimento, modelo_experimento

    collect()
    if usa_cuda:
        torch.cuda.empty_cache()
    return float(resultado["eval_f1_macro"])

In [ ]:
GRUPOS_A_EJECUTAR = ("1", "2", "3", "4", "5")

for grupo, _, _, tasa, epocas in configuraciones_experimento:
    if grupo == "Base" or grupo in GRUPOS_A_EJECUTAR:
        f1_validacion = entrenar_y_evaluar_estilo(tasa, epocas)
        resultados_experimento[grupo] = f1_validacion
        print(f"{grupo} · F1-macro de validación: {f1_validacion:.4f}")

In [ ]:
tabla_experimento = pd.DataFrame([
    {
        "Grupo": grupo,
        "Variable": variable,
        "Valor": valor,
        "F1-macro (validación)": (
            f"{resultados_experimento[grupo]:.4f}" if grupo in resultados_experimento else "Pendiente"
        ),
    }
    for grupo, variable, valor, _, _ in configuraciones_experimento
])
display(tabla_experimento)

### Interpretación del experimento

En esta ejecución, la tasa de **5e-5** obtuvo el mayor F1-macro de validación (**0,9833**), con **118 de 120** aciertos frente a **117 de 120** de la referencia. La diferencia equivale a una frase en esta partición. Con **1e-4**, el F1-macro igualó al de la referencia (**0,9750**).

Con **1e-3**, el F1-macro cayó a **0,3333** y la exactitud al **50 %**. En un conjunto balanceado, estas cifras son compatibles con un clasificador que predice una sola clase.

Con **una época**, el F1-macro fue **0,9667**, ligeramente inferior al de la referencia. Con **seis épocas**, el F1-macro igualó el de la referencia (**0,9750**) y la pérdida de validación subió de **0,0923** a **0,1390**. El aumento de la pérdida es compatible con sobreajuste.

El **0,9750** de la referencia se midió al terminar la tercera época. El **1,0000** de la sección de curvas corresponde al checkpoint de la primera época, seleccionado como el mejor durante la corrida principal de cuatro épocas.

El informe de carga se repite porque cada corrida parte de BETO preentrenado. El estado **UNEXPECTED** corresponde a los pesos de la cabeza de preentrenamiento que se descarta. El estado **MISSING** indica que los pesos del clasificador y del *pooler* se inicializan al cargar el modelo y se ajustan durante el entrenamiento.

## TESTING

Ejecuta la celda, escribe una frase en español y pulsa **Clasificar**. Puedes cambiar la frase y volver a pulsar el botón. BETO muestra el estilo que predice y el porcentaje que asigna a cada clase.

Esta prueba usa el BETO ajustado de la sección 4. Si reiniciaste el kernel, ejecuta las celdas hasta el final de esa sección antes de usar el formulario.

Si tu editor no muestra el formulario, ejecuta la celda de abajo y luego llama a `display(clasificar_frase_beto("Escribe aquí tu frase"))` desde una celda nueva.

In [ ]:
def clasificar_frase_beto(frase):
    frase = frase.strip()
    if not frase:
        raise ValueError("Escribe una frase para clasificar.")

    if "trainer" not in globals() or trainer.state.best_model_checkpoint is None:
        raise RuntimeError("Ejecuta primero el ajuste fino de BETO en la sección 4.")

    beto = trainer.model
    beto.eval()
    dispositivo = next(beto.parameters()).device
    entradas = tokenizer(
        frase, truncation=True, max_length=MAX_TOKENS, return_tensors="pt",
    ).to(dispositivo)

    with torch.inference_mode():
        logits = beto(**entradas).logits[0]
        probabilidades = logits.float().softmax(dim=0).cpu()

    return pd.DataFrame([{
        "Frase": frase,
        "Predicción": ID_A_ESTILO[int(probabilidades.argmax())].capitalize(),
        "Formal (%)": round(100 * probabilidades[ESTILO_A_ID["formal"]].item(), 1),
        "Informal (%)": round(100 * probabilidades[ESTILO_A_ID["informal"]].item(), 1),
    }])


try:
    import ipywidgets as widgets
except ImportError:
    print('Puedes probar una frase con display(clasificar_frase_beto("Tu frase")).')
else:
    campo_frase = widgets.Text(
        description="Frase",
        placeholder="Escribe una frase en español",
        layout=widgets.Layout(width="min(100%, 640px)"),
    )
    boton_clasificar = widgets.Button(description="Clasificar", button_style="primary")
    salida_testing = widgets.Output()

    def mostrar_prediccion(_):
        with salida_testing:
            salida_testing.clear_output(wait=True)
            try:
                resultado = clasificar_frase_beto(campo_frase.value)
            except (ValueError, RuntimeError) as error:
                print(error)
            else:
                display(resultado)

    boton_clasificar.on_click(mostrar_prediccion)
    display(widgets.VBox([campo_frase, boton_clasificar, salida_testing]))

### Referencias

- Material de la clase · **Fine_tuning_emociones_DistilBERT.ipynb**.
- Modelo de lenguaje · [BETO cased de DCC UChile](https://huggingface.co/dccuchile/bert-base-spanish-wwm-cased).
- Implementación del ajuste · [Documentación de Trainer](https://huggingface.co/docs/transformers/en/main_classes/trainer).

## 10. Declaración sobre el uso de inteligencia artificial

Se utilizó **Codex de OpenAI, asistente basado en GPT-6**, como herramienta de apoyo en la elaboración de este notebook sobre clasificación de frases en español según su registro formal o informal.

### Alcance de uso

La intervención de la herramienta incluyó:

- **Desarrollo técnico.** Redacción de las 800 frases sintéticas etiquetadas y escritura del código para la partición por pares, el ajuste fino de BETO, la línea base de TF-IDF con regresión logística y los experimentos de tasa de aprendizaje y número de épocas.
- **Análisis.** Apoyo en el diseño del experimento, la comparación de modelos mediante F1-macro y exactitud, la lectura de las curvas de aprendizaje y de las matrices de confusión, y la interpretación de los errores de clasificación y de los resultados guardados.
- **Redacción y corrección lingüística.** Elaboración y revisión de las explicaciones, el análisis de errores, la conclusión y la interpretación de la tabla de experimentos.
- **Estructura y reproducibilidad.** Organización de las celdas, fijación de la semilla, registro de versiones y comprobaciones del balance de clases, la separación entre conjuntos y la estructura del notebook.

### Responsabilidad técnica y metodológica

La asistencia de Codex abarcó la elaboración del corpus, el código, el análisis y la redacción descritos. El entrenamiento de los modelos y las corridas experimentales se ejecutaron desde el notebook por parte del equipo. Las cifras de las tablas, los informes y las gráficas provienen de las salidas guardadas de esa ejecución. La revisión de las etiquetas, la comprensión del procedimiento y la aceptación de la entrega corresponden al equipo.
